# Projet 12 – Détection automatique de faux billets

## Contexte

L'Organisation nationale de lutte contre le faux-monnayage (ONCFM)
souhaite mettre en place un algorithme capable de différencier
automatiquement les vrais billets des faux billets à partir de leurs
caractéristiques géométriques.

## Objectifs

- Explorer et nettoyer les données disponibles.
- Traiter les éventuelles valeurs manquantes.
- Analyser les caractéristiques des vrais et faux billets.
- Tester plusieurs approches de modélisation :
  - K-means
  - Régression logistique
  - KNN
  - Random Forest
- Comparer leurs performances.
- Analyser les faux positifs et les faux négatifs.
- Sélectionner et justifier le modèle final.
- Développer une application permettant de prédire si un billet est vrai ou faux.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Chargement des données
billets = pd.read_csv("billets.csv", sep=";")

# Affichage des premières lignes
billets.head()

## 1. Exploration et contrôle de la qualité des données

Nous vérifions la structure du jeu de données, les types de variables
et la présence éventuelle de valeurs manquantes.

In [ ]:
# Dimensions du jeu de données
print("Nombre de lignes :", billets.shape[0])
print("Nombre de colonnes :", billets.shape[1])

# Types des variables
print("\nTypes des variables :")
print(billets.dtypes)

# Valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(billets.isnull().sum())

### Interprétation

Le jeu de données contient **1 500 billets décrits par 7 variables** :
une variable cible `is_genuine`, qui indique si le billet est vrai ou faux,
et **6 caractéristiques géométriques numériques** utilisées pour réaliser les prédictions.

Les types de données sont cohérents : les six dimensions sont numériques (`float64`)
et la variable cible est de type booléen (`True` / `False`).

L'analyse des valeurs manquantes montre que seule la variable `margin_low`
contient des données absentes, avec **37 valeurs manquantes sur 1 500 observations**,
soit environ **2,47 % du jeu de données**.

Ces valeurs manquantes devront être traitées avant l'entraînement des modèles
afin de conserver un jeu de données exploitable pour les algorithmes de machine learning.

## 2. Analyse descriptive des données

### 2.1 Répartition des vrais et faux billets

Nous analysons la répartition de la variable cible `is_genuine`
afin d'identifier la proportion de vrais et de faux billets dans le jeu de données.

In [ ]:
# Nombre de vrais et faux billets
repartition = billets["is_genuine"].value_counts()

print("Répartition des billets :")
print(repartition)

# Pourcentages
print("\nRépartition en pourcentage :")
print((billets["is_genuine"].value_counts(normalize=True) * 100).round(2))

# Graphique
repartition.rename(index={True: "Vrais billets", False: "Faux billets"}).plot(
    kind="bar"
)

plt.title("Répartition des vrais et faux billets")
plt.xlabel("Type de billet")
plt.ylabel("Nombre de billets")
plt.xticks(rotation=0)
plt.show()

### Interprétation

Le jeu de données contient **1 000 vrais billets (66,67 %)** et
**500 faux billets (33,33 %)**.

Les vrais billets sont donc **deux fois plus nombreux** que les faux billets.
La variable cible `is_genuine` présente ainsi un **déséquilibre modéré entre les deux classes**.

Ce déséquilibre devra être pris en compte lors de l'évaluation des modèles :
nous ne nous limiterons pas au taux global de bonnes prédictions.
Nous analyserons également la **matrice de confusion**, ainsi que des métriques
comme la précision, le rappel et le F1-score, afin de vérifier notamment
la capacité du modèle à détecter correctement les faux billets.

### 2.2 Statistiques descriptives des caractéristiques géométriques

Nous étudions les principales statistiques descriptives des six caractéristiques
géométriques afin de mieux comprendre leur distribution et leur ordre de grandeur.

In [ ]:
# Sélection des variables géométriques
variables_geo = [
    "diagonal",
    "height_left",
    "height_right",
    "margin_low",
    "margin_up",
    "length"
]

# Statistiques descriptives
billets[variables_geo].describe().round(2)

### Interprétation

Les caractéristiques géométriques des billets présentent des ordres de grandeur différents.
Par exemple, la diagonale moyenne est de **171,96 mm**, tandis que la longueur moyenne
est de **112,68 mm**.

Les hauteurs gauche et droite sont très proches, avec des moyennes respectives de
**104,03 mm** et **103,92 mm**. Les marges présentent des valeurs moyennes de
**4,49 mm pour `margin_low`** et **3,15 mm pour `margin_up`**.

Les écarts-types restent relativement faibles pour plusieurs dimensions, ce qui indique
que les mesures des billets sont globalement concentrées autour de leurs valeurs moyennes.

La variable `margin_low` ne contient que **1 463 observations**, contre **1 500**
pour les autres variables. Cela confirme la présence des **37 valeurs manquantes**
identifiées précédemment.

À ce stade, ces statistiques décrivent l'ensemble des billets. Une comparaison entre
les **vrais et les faux billets** sera nécessaire pour déterminer quelles caractéristiques
permettent le mieux de différencier les deux catégories.

### 2.3 Comparaison des caractéristiques des vrais et faux billets

Nous comparons les valeurs moyennes des caractéristiques géométriques
des vrais et des faux billets afin d'identifier les variables qui semblent
le mieux différencier les deux catégories.

In [ ]:
# Comparaison des moyennes selon le type de billet
moyennes_par_type = billets.groupby("is_genuine")[variables_geo].mean().round(2)

# Renommer les lignes pour faciliter la lecture
moyennes_par_type.index = moyennes_par_type.index.map({
    True: "Vrais billets",
    False: "Faux billets"
})

moyennes_par_type

### Interprétation

La comparaison des moyennes met en évidence plusieurs différences entre les
**vrais et les faux billets**.

La différence la plus marquée concerne la variable `length` :
les vrais billets ont une longueur moyenne de **113,20 mm**, contre
**111,63 mm** pour les faux billets.

La variable `margin_low` présente également une différence importante :
sa moyenne est de **5,22 mm pour les faux billets**, contre **4,12 mm
pour les vrais billets**.

Les faux billets présentent aussi des valeurs moyennes légèrement plus élevées
pour `height_left`, `height_right` et `margin_up`.

À l'inverse, la variable `diagonal` présente des moyennes très proches
(**171,90 mm pour les faux billets contre 171,99 mm pour les vrais billets**).

Ces premiers résultats suggèrent que certaines caractéristiques, notamment
`length` et `margin_low`, pourraient être particulièrement utiles pour
différencier les vrais des faux billets. Cette observation devra cependant
être confirmée par les analyses graphiques et les modèles de classification.

### 2.4 Distribution des principales caractéristiques selon le type de billet

Nous visualisons la distribution de `length` et `margin_low` pour les vrais
et les faux billets afin d'observer si ces caractéristiques permettent
de distinguer graphiquement les deux catégories.

In [ ]:
# Séparation des vrais et faux billets
vrais = billets[billets["is_genuine"] == True]
faux = billets[billets["is_genuine"] == False]

# Distribution de la longueur des billets
plt.figure(figsize=(8, 5))

plt.hist(vrais["length"], bins=30, alpha=0.6, label="Vrais billets")
plt.hist(faux["length"], bins=30, alpha=0.6, label="Faux billets")

plt.title("Distribution de la longueur des billets")
plt.xlabel("Longueur (mm)")
plt.ylabel("Nombre de billets")
plt.legend()
plt.show()

### Interprétation

La distribution de `length` montre une différence nette entre les deux catégories
de billets.

Les **faux billets** sont principalement concentrés autour de longueurs plus faibles,
tandis que les **vrais billets** présentent généralement des longueurs plus élevées.
Cela confirme la différence observée précédemment entre leurs longueurs moyennes
(**111,63 mm pour les faux billets contre 113,20 mm pour les vrais billets**).

Cependant, les deux distributions présentent une **zone de chevauchement**.
La longueur constitue donc une variable potentiellement discriminante, mais elle
ne permet pas à elle seule de séparer parfaitement les vrais des faux billets.

Il sera donc nécessaire de combiner plusieurs caractéristiques géométriques
dans les modèles de machine learning.

### 2.5 Distribution de la marge inférieure (`margin_low`)

Nous analysons la distribution de `margin_low` selon le type de billet.
Cette variable présente 37 valeurs manquantes qui seront traitées
avant l'entraînement des modèles.

In [ ]:
# Distribution de margin_low selon le type de billet
plt.figure(figsize=(8, 5))

plt.hist(
    vrais["margin_low"].dropna(),
    bins=30,
    alpha=0.6,
    label="Vrais billets"
)

plt.hist(
    faux["margin_low"].dropna(),
    bins=30,
    alpha=0.6,
    label="Faux billets"
)

plt.title("Distribution de la marge inférieure des billets")
plt.xlabel("Marge inférieure (mm)")
plt.ylabel("Nombre de billets")
plt.legend()
plt.show()

### Interprétation

La distribution de `margin_low` met en évidence une différence importante
entre les vrais et les faux billets.

Les **vrais billets** présentent généralement une marge inférieure plus faible,
avec une moyenne de **4,12 mm**, tandis que les **faux billets** ont une marge
inférieure plus élevée, avec une moyenne de **5,22 mm**.

Les deux distributions présentent néanmoins une **zone de chevauchement**,
principalement autour de valeurs comprises entre environ **4 et 5 mm**.

La variable `margin_low` semble donc apporter une information utile pour
différencier les deux catégories de billets, mais elle ne permet pas à elle
seule une séparation parfaite.

Combinée à `length` et aux autres caractéristiques géométriques,
elle pourra contribuer à améliorer les performances des modèles de classification.

## 3. Traitement des valeurs manquantes

La variable `margin_low` contient 37 valeurs manquantes.
Avant de choisir une méthode d'imputation, nous analysons leur répartition
entre les vrais et les faux billets.

### 3.1 Analyse des valeurs manquantes

In [ ]:
## Identification des billets dont margin_low est manquant
manquants_margin_low = billets[billets["margin_low"].isna()]

print("Nombre total de valeurs manquantes :", len(manquants_margin_low))

print("\nRépartition selon le type de billet :")
print(manquants_margin_low["is_genuine"].value_counts())

### Interprétation

Les **37 valeurs manquantes** de `margin_low` concernent les deux catégories
de billets : **29 vrais billets** et **8 faux billets**.

La majorité des valeurs manquantes appartient donc aux vrais billets.
Cependant, le nombre total de valeurs manquantes reste faible par rapport
aux **1 500 observations** du jeu de données.

Plutôt que de supprimer ces observations et de perdre de l'information,
nous allons conserver les billets concernés et **imputer les valeurs manquantes**.

L'imputation devra être réalisée sans utiliser directement la variable cible
`is_genuine`, afin d'éviter d'introduire artificiellement dans les données
une information liée à la classe que les modèles devront ensuite prédire.

In [ ]:
# Moyenne et médiane de margin_low
moyenne_margin_low = billets["margin_low"].mean()
mediane_margin_low = billets["margin_low"].median()

print("Moyenne de margin_low :", round(moyenne_margin_low, 2), "mm")
print("Médiane de margin_low :", round(mediane_margin_low, 2), "mm")

### 3.2 Choix de la méthode d'imputation

La moyenne de `margin_low` est de **4,49 mm** et sa médiane de **4,31 mm**.

La distribution de cette variable présente une certaine asymétrie et quelques
valeurs élevées. Nous choisissons donc une **imputation par la médiane**,
car celle-ci est moins sensible aux valeurs extrêmes que la moyenne.

Cette méthode permet également de conserver les 37 observations concernées
plutôt que de les supprimer.

### 3.3 Imputation et vérification des valeurs manquantes

In [ ]:
# Les valeurs manquantes seront imputées après la séparation train/test.
# La médiane sera calculée uniquement sur le jeu d'entraînement
# afin d'éviter toute fuite d'information.

print("Valeurs manquantes de margin_low avant séparation :",
      billets["margin_low"].isna().sum())

### Interprétation

La variable `margin_low` contient **37 valeurs manquantes** sur les **1 500 observations**.

À ce stade, ces valeurs ne sont volontairement pas encore imputées. Le jeu de données sera d'abord séparé en un **jeu d'entraînement** et un **jeu de test**.

L'imputation sera ensuite réalisée en calculant la médiane de `margin_low` **uniquement sur le jeu d'entraînement**, puis en utilisant cette même valeur pour compléter les données d'entraînement et de test.

Cette méthode permet d'éviter toute **fuite d'information provenant du jeu de test**.

## 4. Analyse des relations entre les variables

Nous étudions les relations entre les différentes caractéristiques géométriques
des billets afin d'identifier d'éventuelles corrélations entre les variables.

### 4.1 Matrice de corrélation

La matrice de corrélation permet de mesurer le sens et l'intensité des relations
linéaires entre les différentes caractéristiques géométriques des billets.

In [ ]:
# Calcul de la matrice de corrélation
correlation = billets[variables_geo].corr().round(2)

# Affichage de la matrice
correlation

### Interprétation

La matrice de corrélation montre que certaines caractéristiques géométriques
présentent des relations entre elles.

La relation la plus marquée concerne `margin_low` et `length`, avec une
corrélation de **-0,66**. Cette corrélation négative indique que lorsque
`margin_low` augmente, `length` a tendance à diminuer.

Une corrélation négative relativement importante apparaît également entre
`margin_up` et `length` (**-0,52**).

À l'inverse, `margin_low` et `margin_up` présentent une corrélation positive
de **0,43** : elles ont donc tendance à évoluer dans le même sens.

Les variables `height_left` et `height_right` sont modérément associées à
certaines autres dimensions, tandis que `diagonal` présente des corrélations
faibles avec l'ensemble des autres caractéristiques.

Aucune corrélation n'est cependant extrêmement proche de **+1 ou -1**.
Les variables apportent donc des informations différentes qui pourront être
combinées lors de la construction des modèles de classification.

### 4.2 Visualisation de la matrice de corrélation

Nous représentons graphiquement la matrice de corrélation afin de faciliter
l'identification des relations les plus importantes entre les variables.

In [ ]:
# Visualisation de la matrice de corrélation
plt.figure(figsize=(8, 6))

plt.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)

plt.colorbar(label="Coefficient de corrélation")

plt.xticks(
    range(len(correlation.columns)),
    correlation.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(correlation.index)),
    correlation.index
)

# Affichage des coefficients dans les cases
for i in range(len(correlation.index)):
    for j in range(len(correlation.columns)):
        plt.text(
            j, i,
            correlation.iloc[i, j],
            ha="center",
            va="center"
        )

plt.title("Matrice de corrélation des caractéristiques géométriques")
plt.tight_layout()
plt.show()

### Interprétation

La représentation graphique confirme les résultats de la matrice de corrélation.

Les relations les plus marquées concernent principalement `length` avec
`margin_low` (**-0,66**) et `margin_up` (**-0,52**).

La heatmap montre également qu'aucune paire de caractéristiques ne présente
une corrélation extrêmement forte proche de **+1 ou -1**.

Nous conservons donc les six caractéristiques géométriques pour la suite
de la modélisation.

## 5. Préparation des données pour le Machine Learning

Après le nettoyage et l'analyse exploratoire, nous préparons les données
qui seront utilisées pour entraîner et évaluer les différents modèles.

Les six caractéristiques géométriques constituent les variables explicatives (`X`),
tandis que `is_genuine` constitue la variable cible (`y`).

### 5.1 Séparation des variables explicatives et de la variable cible

In [ ]:
# Variables explicatives
X = billets[variables_geo]

# Variable cible
y = billets["is_genuine"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nVariables explicatives :")
print(list(X.columns))

print("\nRépartition de la variable cible :")
print(y.value_counts())

### Interprétation

Les variables explicatives `X` contiennent **1 500 observations et 6 caractéristiques
géométriques** : `diagonal`, `height_left`, `height_right`, `margin_low`,
`margin_up` et `length`.

La variable cible `y` contient **1 500 valeurs** indiquant si chaque billet
est vrai (`True`) ou faux (`False`).

Les données sont maintenant séparées entre les caractéristiques utilisées
pour la prédiction et la variable que les modèles devront prédire.

### 5.2 Séparation des données d'entraînement et de test

Pour évaluer les modèles sur des données qu'ils n'ont jamais vues pendant
leur apprentissage, le jeu de données est séparé en deux parties :

- **80 %** des observations pour l'entraînement ;
- **20 %** des observations pour le test.

Une séparation stratifiée est utilisée afin de conserver la proportion
de vrais et de faux billets dans les deux ensembles.

In [ ]:
# Import de la fonction de séparation
from sklearn.model_selection import train_test_split

# Séparation en jeux d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Vérification des dimensions
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

# Vérification de la répartition des classes
print("\nRépartition dans y_train :")
print(y_train.value_counts())

print("\nRépartition dans y_test :")
print(y_test.value_counts())

### Interprétation

Le jeu de données a été séparé en deux ensembles :

- **1 200 billets (80 %)** sont utilisés pour entraîner les modèles ;
- **300 billets (20 %)** sont conservés pour évaluer leurs performances.

Grâce à la stratification, la répartition des classes est conservée dans
les deux ensembles.

Le jeu d'entraînement contient **800 vrais billets et 400 faux billets**,
tandis que le jeu de test contient **200 vrais billets et 100 faux billets**.

Cette séparation permettra d'évaluer les modèles sur des données qu'ils
n'auront pas utilisées pendant leur apprentissage.

In [ ]:
# Imputation des valeurs manquantes après la séparation train/test
# La médiane est calculée uniquement sur le jeu d'entraînement

mediane_margin_low_train = X_train["margin_low"].median()

print("Médiane de margin_low calculée sur X_train :",
      round(mediane_margin_low_train, 2), "mm")

# Imputation dans le jeu d'entraînement
X_train = X_train.copy()
X_train["margin_low"] = X_train["margin_low"].fillna(
    mediane_margin_low_train
)

# Imputation dans le jeu de test avec la médiane du train
X_test = X_test.copy()
X_test["margin_low"] = X_test["margin_low"].fillna(
    mediane_margin_low_train
)

# Vérification
print("\nValeurs manquantes après imputation :")
print("X_train :", X_train.isna().sum().sum())
print("X_test  :", X_test.isna().sum().sum())

### Interprétation

La médiane de `margin_low`, calculée **uniquement sur le jeu d'entraînement**, est de **4,32 mm**.

Cette valeur a été utilisée pour imputer les valeurs manquantes de `margin_low` dans le **jeu d'entraînement** et dans le **jeu de test**.

Après imputation, `X_train` et `X_test` ne contiennent plus **aucune valeur manquante**.

Le calcul de la médiane uniquement à partir des données d'entraînement permet d'éviter toute **fuite d'information provenant du jeu de test**.

### 5.3 Standardisation des variables

Les caractéristiques géométriques des billets ne sont pas toutes exprimées
sur les mêmes échelles.

Nous standardisons donc les variables afin qu'elles soient comparables et
qu'une caractéristique ayant des valeurs plus élevées ne domine pas
artificiellement les autres.

Le `StandardScaler` est ajusté uniquement sur le jeu d'entraînement afin
d'éviter toute fuite d'information provenant du jeu de test.

In [ ]:
# Import du StandardScaler
from sklearn.preprocessing import StandardScaler

# Création du scaler
scaler = StandardScaler()

# Ajustement uniquement sur les données d'entraînement
X_train_scaled = scaler.fit_transform(X_train)

# Transformation des données de test avec le même scaler
X_test_scaled = scaler.transform(X_test)

# Vérification des dimensions
print("Dimensions de X_train_scaled :", X_train_scaled.shape)
print("Dimensions de X_test_scaled  :", X_test_scaled.shape)

# Vérification de la moyenne et de l'écart-type
print("\nMoyennes après standardisation :")
print(X_train_scaled.mean(axis=0).round(2))

print("\nÉcarts-types après standardisation :")
print(X_train_scaled.std(axis=0).round(2))

### Interprétation

La standardisation a conservé les dimensions des jeux de données :
**1 200 observations pour l'entraînement** et **300 observations pour le test**,
avec **6 caractéristiques géométriques**.

Après transformation, les moyennes des variables du jeu d'entraînement sont
proches de **0** et leurs écarts-types sont égaux à **1**.

Les caractéristiques sont désormais placées sur une échelle comparable,
ce qui évite qu'une variable influence davantage les modèles uniquement
à cause de son ordre de grandeur.

Le scaler ayant été ajusté uniquement sur le jeu d'entraînement, aucune
information provenant du jeu de test n'a été utilisée lors de cette étape.

## 6. Modélisation par K-means

In [ ]:
# 6.1.1 Entraînement du modèle K-means

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")

### 6.1.2 Principe et entraînement du modèle K-means

K-means est un algorithme d'apprentissage **non supervisé** qui regroupe
les observations en fonction de leurs similarités.

Dans notre cas, nous demandons à l'algorithme de former **2 clusters**,
car nous cherchons à distinguer deux groupes de billets.

K-means n'utilise pas la variable cible `is_genuine` pour créer les groupes.
Nous comparerons ensuite les clusters obtenus aux vraies catégories
afin d'évaluer la qualité de la séparation.

In [ ]:
# Import de KMeans
from sklearn.cluster import KMeans

# Création du modèle avec 2 clusters
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

# Entraînement sur les données standardisées
kmeans.fit(X_train_scaled)

# Attribution d'un cluster à chaque billet du jeu d'entraînement
clusters_train = kmeans.labels_

# Nombre de billets dans chaque cluster
import numpy as np

valeurs, effectifs = np.unique(clusters_train, return_counts=True)

print("Répartition des billets par cluster :")
for cluster, effectif in zip(valeurs, effectifs):
    print(f"Cluster {cluster} : {effectif} billets")

### Interprétation

K-means a réparti les **1 200 billets du jeu d'entraînement** en deux groupes :

- **Cluster 0 : 394 billets**
- **Cluster 1 : 806 billets**

Cette répartition est proche de celle des classes réelles du jeu d'entraînement,
qui contient **400 faux billets et 800 vrais billets**.

Cependant, les numéros des clusters sont attribués arbitrairement par K-means.
À ce stade, nous ne pouvons donc pas conclure que le cluster 0 correspond
aux faux billets et le cluster 1 aux vrais billets.

Nous devons comparer les clusters obtenus avec les catégories réelles
`is_genuine` pour déterminer leur correspondance.

### 6.2 Correspondance entre les clusters et les vrais/faux billets

Nous comparons maintenant les clusters créés par K-means avec les catégories
réelles des billets afin de déterminer à quel type de billet correspond
chaque cluster.

In [ ]:
# Comparaison entre les clusters et les vraies catégories
comparaison_kmeans = pd.crosstab(
    y_train,
    clusters_train,
    rownames=["Type réel"],
    colnames=["Cluster K-means"]
)

comparaison_kmeans

### Interprétation

La comparaison avec les catégories réelles permet d'identifier clairement
la correspondance entre les deux clusters :

- le **cluster 0** correspond principalement aux **faux billets** :
  388 faux billets contre seulement 6 vrais billets ;
- le **cluster 1** correspond principalement aux **vrais billets** :
  794 vrais billets contre seulement 12 faux billets.

Sur les **1 200 billets** du jeu d'entraînement, **1 182** sont ainsi regroupés
dans le cluster correspondant à leur catégorie réelle, contre **18 billets**
placés dans l'autre groupe.

K-means parvient donc à faire apparaître naturellement deux groupes très proches
des catégories réelles, alors que la variable cible `is_genuine` n'a pas été
utilisée pour créer les clusters.

Ce résultat montre que les caractéristiques géométriques contiennent une
structure permettant de distinguer efficacement les vrais des faux billets.

### 6.3 Évaluation de K-means sur le jeu de test

Après avoir identifié la correspondance entre les clusters et les catégories
réelles sur le jeu d'entraînement, nous évaluons maintenant K-means sur
les **300 billets du jeu de test**.

Le cluster 0 est associé aux **faux billets** (`False`) et le cluster 1
aux **vrais billets** (`True`).

In [ ]:
# Prédiction des clusters sur le jeu de test
clusters_test = kmeans.predict(X_test_scaled)

# Correspondance identifiée sur le jeu d'entraînement :
# Cluster 0 = Faux (False)
# Cluster 1 = Vrai (True)
y_pred_kmeans = clusters_test.astype(bool)

# Comparaison entre valeurs réelles et prédictions
comparaison_test_kmeans = pd.crosstab(
    y_test,
    y_pred_kmeans,
    rownames=["Type réel"],
    colnames=["Prédiction K-means"]
)

print("Comparaison sur le jeu de test :")
comparaison_test_kmeans

### Interprétation

Sur les **300 billets du jeu de test**, K-means classe correctement
**296 billets** et commet seulement **4 erreurs**.

Parmi les **100 faux billets**, 98 sont correctement associés à la catégorie
des faux billets et 2 sont classés comme vrais.

Parmi les **200 vrais billets**, 198 sont correctement associés à la catégorie
des vrais billets et 2 sont classés comme faux.

Ces premiers résultats montrent que K-means généralise bien sur les données
de test. Nous allons maintenant calculer les métriques de performance afin
de quantifier précisément cette qualité de classification.

### 6.4 Matrice de confusion et métriques de performance

Nous évaluons maintenant les performances de K-means à l'aide de plusieurs
indicateurs : l'accuracy, la précision, le rappel et le F1-score.

La matrice de confusion permet également d'identifier précisément les
classifications correctes et les erreurs commises par le modèle.

In [ ]:
# Import des métriques d'évaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Calcul des métriques
accuracy_kmeans = accuracy_score(y_test, y_pred_kmeans)
precision_kmeans = precision_score(y_test, y_pred_kmeans)
recall_kmeans = recall_score(y_test, y_pred_kmeans)
f1_kmeans = f1_score(y_test, y_pred_kmeans)

# Matrice de confusion
cm_kmeans = confusion_matrix(y_test, y_pred_kmeans)

print("Matrice de confusion :")
print(cm_kmeans)

print("\nAccuracy :", round(accuracy_kmeans, 4))
print("Précision :", round(precision_kmeans, 4))
print("Rappel :", round(recall_kmeans, 4))
print("F1-score :", round(f1_kmeans, 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_kmeans,
    target_names=["Faux billets", "Vrais billets"]
))

### Interprétation

K-means obtient de très bonnes performances sur le jeu de test avec une
**accuracy de 98,67 %** : **296 billets sur 300** sont correctement classés.

La matrice de confusion montre que :

- parmi les **100 faux billets**, **98 sont correctement identifiés** et
  **2 sont classés à tort comme vrais** ;
- parmi les **200 vrais billets**, **198 sont correctement identifiés** et
  **2 sont classés à tort comme faux**.

Pour les **faux billets**, la précision, le rappel et le F1-score atteignent
environ **98 %**. Pour les **vrais billets**, ces indicateurs atteignent
environ **99 %**.

Ces résultats montrent que les caractéristiques géométriques permettent à
K-means de former des groupes très proches des catégories réelles, malgré
le fait qu'il s'agisse d'un algorithme d'apprentissage non supervisé.

K-means constitue donc une première solution très performante. Il devra
cependant être comparé aux modèles supervisés avant de sélectionner le
modèle final.

## 7. Modélisation par régression logistique

### 7.1 Principe et entraînement de la régression logistique

La régression logistique est un algorithme de classification **supervisé**.

Contrairement à K-means, elle utilise pendant son apprentissage les
caractéristiques géométriques des billets (`X_train`) ainsi que leur
catégorie réelle (`y_train`).

L'objectif est d'apprendre à estimer la probabilité qu'un billet appartienne
à l'une des deux catégories : **vrai** ou **faux**.

Le modèle est entraîné sur les données standardisées afin que les différentes
caractéristiques soient placées sur une échelle comparable.

In [ ]:
# Import de la régression logistique
from sklearn.linear_model import LogisticRegression

# Création du modèle
logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

# Entraînement du modèle
logistic_model.fit(X_train_scaled, y_train)

# Prédictions sur le jeu de test
y_pred_logistic = logistic_model.predict(X_test_scaled)

print("Nombre de prédictions :", len(y_pred_logistic))

print("\nRépartition des prédictions :")
print(pd.Series(y_pred_logistic).value_counts())

### Interprétation

La régression logistique a produit une prédiction pour chacun des
**300 billets du jeu de test**.

Le modèle prédit :

- **201 vrais billets** (`True`) ;
- **99 faux billets** (`False`).

Cette répartition est très proche de la répartition réelle du jeu de test,
qui contient **200 vrais billets et 100 faux billets**.

Cependant, cette seule répartition ne permet pas encore d'évaluer la qualité
du modèle. Il est nécessaire de comparer les prédictions aux catégories
réelles à l'aide d'une matrice de confusion et de métriques de performance.

In [ ]:

# Matrice de confusion - Régression logistique ONCFM

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Calcul de la matrice de confusion
cm = confusion_matrix(
    y_test,
    y_pred_logistic,
    labels=[False, True]
)

# Création du graphique
fig, ax = plt.subplots(figsize=(7, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Faux billet", "Vrai billet"]
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("Matrice de confusion - Régression logistique ONCFM")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.tight_layout()

# Enregistrement pour PowerPoint
plt.savefig(
    "matrice_confusion_ONCFM.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Graphique enregistré avec succès !")


### 7.2 Évaluation des performances de la régression logistique

Nous comparons les prédictions de la régression logistique aux catégories
réelles du jeu de test.

Nous utilisons la matrice de confusion ainsi que l'accuracy, la précision,
le rappel et le F1-score afin d'évaluer les performances du modèle.

In [ ]:
# Calcul des métriques de la régression logistique
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
precision_logistic = precision_score(y_test, y_pred_logistic)
recall_logistic = recall_score(y_test, y_pred_logistic)
f1_logistic = f1_score(y_test, y_pred_logistic)

# Matrice de confusion
cm_logistic = confusion_matrix(y_test, y_pred_logistic)

print("Matrice de confusion :")
print(cm_logistic)

print("\nAccuracy :", round(accuracy_logistic, 4))
print("Précision :", round(precision_logistic, 4))
print("Rappel :", round(recall_logistic, 4))
print("F1-score :", round(f1_logistic, 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_logistic,
    target_names=["Faux billets", "Vrais billets"]
))

### Interprétation

La régression logistique obtient de très bonnes performances sur le jeu de test, avec une **accuracy de 99 %**, soit **297 billets correctement classés sur 300**.

La matrice de confusion montre que :

- parmi les **100 faux billets**, **98 sont correctement identifiés** et **2 sont classés à tort comme vrais** ;
- parmi les **200 vrais billets**, **199 sont correctement identifiés** et **1 seul est classé à tort comme faux**.

Le rapport de classification confirme ces bonnes performances, avec des valeurs de **précision, rappel et F1-score proches de 98 % pour les faux billets et de 99 % pour les vrais billets**.

Ces résultats montrent que la régression logistique généralise très bien sur des données qui n'ont pas été utilisées pendant son apprentissage.

Comparée à K-means, elle commet **3 erreurs contre 4 pour K-means** sur le même jeu de test. Elle présente donc, à ce stade, une performance globale légèrement supérieure.


### 7.3 Analyse des probabilités de prédiction

In [ ]:
# 7.3.1 Calcul des probabilités de prédiction

# Probabilité d'appartenir à chaque classe
probabilites_logistic = logistic_model.predict_proba(X_test_scaled)

# Création d'un tableau pour visualiser quelques résultats
resultats_probabilites = pd.DataFrame({
    "Probabilité Faux": probabilites_logistic[:, 0],
    "Probabilité Vrai": probabilites_logistic[:, 1],
    "Prédiction": y_pred_logistic,
    "Valeur réelle": y_test.values
})

# Affichage des 10 premières observations
resultats_probabilites.head(10).round(4)

### Interprétation

Les probabilités de prédiction permettent d'évaluer le niveau de confiance
de la régression logistique pour chaque billet.

Pour la majorité des billets affichés, le modèle attribue une probabilité
très élevée à la classe prédite. Par exemple, certains vrais billets sont
identifiés avec une probabilité supérieure à **99 %**, tandis que certains
faux billets présentent une probabilité proche de **100 % d'être faux**.

Certaines observations sont toutefois plus proches de la frontière de décision.
Par exemple, l'observation 3 est classée comme un **vrai billet** avec une
probabilité de **63,88 %**, contre **36,12 %** pour la classe faux billet.

Ces résultats montrent que la régression logistique permet non seulement
d'obtenir une classification Vrai/Faux, mais également d'estimer le
**niveau de confiance associé à chaque prédiction**.

### 7.3.2 Analyse des prédictions erronées

Nous analysons les billets mal classés par la régression logistique afin
d'identifier le niveau de confiance du modèle lorsqu'il commet une erreur.

In [ ]:
# 7.3.2.1 Identification des prédictions erronées

# Sélection des billets pour lesquels la prédiction est différente de la valeur réelle
erreurs_logistic = resultats_probabilites[
    resultats_probabilites["Prédiction"] != resultats_probabilites["Valeur réelle"]
]

print("Nombre de prédictions erronées :", len(erreurs_logistic))

# Affichage des erreurs et de leurs probabilités
erreurs_logistic.round(4)

### Interprétation

La régression logistique commet seulement **3 erreurs sur les 300 billets**
du jeu de test, soit **1 % des observations**.

L'analyse des probabilités montre cependant que ces erreurs présentent
des niveaux de confiance différents :

- le billet **65** est réellement un vrai billet, mais le modèle le classe
  comme faux avec une probabilité de **92,52 %**. Il s'agit donc d'une erreur
  commise avec un niveau de confiance élevé ;

- le billet **84** est réellement un faux billet, mais il est classé comme
  vrai avec une probabilité de **52,07 %**. Cette observation est très proche
  du seuil de décision de 50 %, ce qui traduit une forte incertitude du modèle ;

- le billet **206** est également un faux billet classé comme vrai, avec une
  probabilité de **67,55 %** d'être vrai.

Parmi les trois erreurs, **deux faux billets sont donc considérés à tort comme
vrais**. Ce type d'erreur est particulièrement important dans le contexte de
l'ONCFM, puisque l'objectif principal est de détecter les billets contrefaits.

Malgré ces quelques erreurs, la régression logistique conserve une excellente
performance globale avec **99 % d'accuracy** sur le jeu de test.

# 8. Modélisation par KNN

Le modèle **K-Nearest Neighbors (KNN)** classe un nouveau billet en fonction
des billets les plus proches dans l'espace des caractéristiques géométriques.

Comme cet algorithme repose sur des calculs de distance, nous utilisons les
données standardisées précédemment afin que toutes les variables contribuent
de manière comparable à la classification.

## 8.1 Entraînement du modèle KNN

In [ ]:
# 8.1.1 Entraînement du modèle KNN

from sklearn.neighbors import KNeighborsClassifier

# Création du modèle avec 5 voisins
knn_model = KNeighborsClassifier(n_neighbors=5)

# Entraînement sur le jeu d'entraînement standardisé
knn_model.fit(X_train_scaled, y_train)

# Prédictions sur le jeu de test
y_pred_knn = knn_model.predict(X_test_scaled)

print("Nombre de prédictions :", len(y_pred_knn))
print("\nRépartition des prédictions :")
print(pd.Series(y_pred_knn).value_counts())

### Interprétation

Le modèle KNN a produit une prédiction pour chacun des **300 billets**
du jeu de test.

Il prédit :

- **201 vrais billets** ;
- **99 faux billets**.

Cette répartition est très proche de la répartition réelle du jeu de test,
qui contient **200 vrais billets et 100 faux billets**.

Cependant, cette répartition ne permet pas à elle seule d'évaluer les
performances du modèle. Nous devons maintenant comparer les prédictions
aux valeurs réelles à l'aide de la matrice de confusion et des métriques
de classification.

## 8.2 Évaluation des performances du KNN

In [ ]:
# 8.2.1 Évaluation des performances du modèle KNN

# Calcul des métriques
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

# Matrice de confusion
cm_knn = confusion_matrix(y_test, y_pred_knn)

print("Matrice de confusion :")
print(cm_knn)

print("\nAccuracy :", round(accuracy_knn, 4))
print("Précision :", round(precision_knn, 4))
print("Rappel :", round(recall_knn, 4))
print("F1-score :", round(f1_knn, 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_knn,
    target_names=["Faux billets", "Vrais billets"]
))

### Interprétation

Le modèle KNN obtient de très bonnes performances sur le jeu de test,
avec une **accuracy de 98,33 %**.

La matrice de confusion montre que :

- parmi les **100 faux billets**, **97 sont correctement identifiés**
  et **3 sont classés à tort comme vrais** ;
- parmi les **200 vrais billets**, **198 sont correctement identifiés**
  et **2 sont classés à tort comme faux**.

Le modèle commet donc **5 erreurs sur 300 billets**.

Pour les faux billets, le rappel atteint **97 %**. Cette métrique est
particulièrement importante dans le contexte de l'ONCFM, car elle mesure
la capacité du modèle à détecter effectivement les billets contrefaits.

Avec **k = 5**, KNN présente donc d'excellentes performances, mais il est
à ce stade légèrement moins performant que la régression logistique,
qui obtenait **99 % d'accuracy** avec seulement **3 erreurs**.

## 8.3 Choix du nombre optimal de voisins (k)

Le nombre de voisins `k` est un hyperparamètre important du modèle KNN.

Afin de ne pas choisir cette valeur arbitrairement, nous comparons plusieurs
valeurs de `k` à l'aide d'une validation croisée réalisée uniquement sur
le jeu d'entraînement.

Cette méthode permet de sélectionner le nombre de voisins offrant la meilleure
capacité de généralisation sans utiliser le jeu de test pour régler le modèle.

In [ ]:
# 8.3.1 Recherche du nombre optimal de voisins par validation croisée

from sklearn.model_selection import cross_val_score

# Valeurs de k à tester
valeurs_k = range(1, 21)

scores_moyens = []

# Test de chaque valeur de k
for k in valeurs_k:
    modele_knn = KNeighborsClassifier(n_neighbors=k)
    
    scores = cross_val_score(
        modele_knn,
        X_train_scaled,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    
    scores_moyens.append(scores.mean())

# Meilleure valeur de k
meilleur_k = list(valeurs_k)[np.argmax(scores_moyens)]
meilleur_score = max(scores_moyens)

print("Meilleur nombre de voisins (k) :", meilleur_k)
print("Accuracy moyenne en validation croisée :", round(meilleur_score, 4))

### Interprétation

La validation croisée à 5 plis a permis de comparer plusieurs valeurs du
nombre de voisins `k`, comprises entre **1 et 20**.

La meilleure performance moyenne est obtenue avec **k = 12**, avec une
accuracy moyenne de **99,25 %** sur les données de validation.

Cette valeur a été sélectionnée uniquement à partir du jeu d'entraînement,
sans utiliser le jeu de test. Cela permet de limiter le risque de choisir
un hyperparamètre spécifiquement adapté aux données de test.

Nous retenons donc **k = 12** pour construire la version optimisée du modèle KNN.

## 8.4 Évaluation du KNN optimisé

In [ ]:
# 8.4.1 Entraînement et évaluation du KNN optimisé

# Création du modèle avec le meilleur k obtenu
knn_optimise = KNeighborsClassifier(n_neighbors=meilleur_k)

# Entraînement sur l'ensemble du jeu d'entraînement
knn_optimise.fit(X_train_scaled, y_train)

# Prédictions sur le jeu de test
y_pred_knn_optimise = knn_optimise.predict(X_test_scaled)

# Calcul des métriques
accuracy_knn_optimise = accuracy_score(y_test, y_pred_knn_optimise)
precision_knn_optimise = precision_score(y_test, y_pred_knn_optimise)
recall_knn_optimise = recall_score(y_test, y_pred_knn_optimise)
f1_knn_optimise = f1_score(y_test, y_pred_knn_optimise)

# Matrice de confusion
cm_knn_optimise = confusion_matrix(y_test, y_pred_knn_optimise)

print("Meilleur k utilisé :", meilleur_k)

print("\nMatrice de confusion :")
print(cm_knn_optimise)

print("\nAccuracy :", round(accuracy_knn_optimise, 4))
print("Précision :", round(precision_knn_optimise, 4))
print("Rappel :", round(recall_knn_optimise, 4))
print("F1-score :", round(f1_knn_optimise, 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_knn_optimise,
    target_names=["Faux billets", "Vrais billets"]
))

### Interprétation

L'optimisation du nombre de voisins améliore les performances du modèle KNN.

Avec **k = 12**, le modèle atteint une **accuracy de 99 %** sur les **300 billets du jeu de test**, soit **297 billets correctement classés sur 300**.

La matrice de confusion montre que :

- parmi les **100 faux billets**, **98 sont correctement identifiés** et **2 sont classés à tort comme vrais** ;
- parmi les **200 vrais billets**, **199 sont correctement identifiés** et **1 seul est classé à tort comme faux**.

Le modèle commet donc seulement **3 erreurs sur 300 billets**.

Le passage de **k = 5 à k = 12** améliore l'accuracy de **98,33 % à 99 %** et réduit le nombre d'erreurs de **5 à 3**.

La valeur **k = 12** a été sélectionnée par **validation croisée sur le jeu d'entraînement**, avec une accuracy moyenne de **99,25 %**, sans utiliser le jeu de test pour choisir cet hyperparamètre.

Sur ce jeu de test, le **KNN optimisé obtient la même matrice de confusion et la même accuracy de 99 % que la régression logistique**.

Ces résultats confirment l'intérêt de sélectionner l'hyperparamètre **k par validation croisée** plutôt que de choisir arbitrairement sa valeur.


# 9. Modélisation par Random Forest

Le modèle Random Forest combine plusieurs arbres de décision afin de produire
une classification plus robuste.

Contrairement à KNN, il ne repose pas directement sur des calculs de distance.
Il permet également d'étudier l'importance des différentes caractéristiques
géométriques dans la prédiction du type de billet.

## 9.1 Entraînement du modèle Random Forest

Nous entraînons un modèle Random Forest sur le jeu d'entraînement.

Le modèle combine plusieurs arbres de décision et réalise la prédiction finale
à partir des décisions prises par l'ensemble des arbres.

In [ ]:
# 9.1.1 Entraînement du modèle Random Forest

from sklearn.ensemble import RandomForestClassifier

# Création du modèle
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Entraînement du modèle
random_forest_model.fit(X_train, y_train)

# Prédictions sur le jeu de test
y_pred_rf = random_forest_model.predict(X_test)

print("Nombre de prédictions :", len(y_pred_rf))

print("\nRépartition des prédictions :")
print(pd.Series(y_pred_rf).value_counts())

### Interprétation

Le modèle Random Forest a produit une prédiction pour chacun des
**300 billets du jeu de test**.

Il prédit :

- **201 vrais billets** ;
- **99 faux billets**.

Cette répartition est très proche de la répartition réelle du jeu de test,
qui contient **200 vrais billets et 100 faux billets**.

Il est maintenant nécessaire de comparer les prédictions aux valeurs réelles
à l'aide de la matrice de confusion et des principales métriques de
classification.

In [ ]:

# Graphique - Importance des variables Random Forest ONCFM

import pandas as pd
import matplotlib.pyplot as plt

# Recherche du modèle Random Forest déjà entraîné
if "rf_model" in globals():
    modele_rf = rf_model
elif "random_forest_model" in globals():
    modele_rf = random_forest_model
else:
    raise NameError(
        "Modèle Random Forest introuvable. "
        "Vérifie son nom dans la cellule d'entraînement."
    )

# Récupération des importances calculées par le modèle
importances = pd.Series(
    modele_rf.feature_importances_ * 100,
    index=X_train.columns
).sort_values(ascending=True)

# Création du graphique
fig, ax = plt.subplots(figsize=(9, 5))

barres = ax.barh(
    importances.index,
    importances.values,
    color="steelblue"
)

# Affichage des pourcentages
for barre in barres:
    valeur = barre.get_width()
    ax.text(
        valeur + 0.5,
        barre.get_y() + barre.get_height() / 2,
        f"{valeur:.2f} %",
        va="center"
    )

ax.set_xlim(0, max(importances.values) * 1.18)
ax.set_xlabel("Importance des variables (%)")
ax.set_ylabel("Caractéristiques géométriques")
ax.set_title(
    "Importance des variables - Random Forest ONCFM"
)

plt.tight_layout()

# Sauvegarde du graphique
plt.savefig(
    "importance_variables_ONCFM.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Graphique Random Forest enregistré avec succès !")
print(importances.sort_values(ascending=False).round(2))


## 9.2 Évaluation des performances du Random Forest

In [ ]:
# 9.2.1 Évaluation des performances du Random Forest

# Calcul des métriques
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

# Matrice de confusion
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Matrice de confusion :")
print(cm_rf)

print("\nAccuracy :", round(accuracy_rf, 4))
print("Précision :", round(precision_rf, 4))
print("Rappel :", round(recall_rf, 4))
print("F1-score :", round(f1_rf, 4))

print("\nRapport de classification :")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["Faux billets", "Vrais billets"]
))

### Interprétation

Le modèle Random Forest obtient d'excellentes performances sur le jeu de test,
avec une **accuracy de 99 %**.

La matrice de confusion montre que :

- parmi les **100 faux billets**, **98 sont correctement identifiés**
  et **2 sont classés à tort comme vrais** ;
- parmi les **200 vrais billets**, **199 sont correctement identifiés**
  et **1 seul est classé à tort comme faux**.

Le modèle commet donc seulement **3 erreurs sur les 300 billets** du jeu de test.

Le rappel atteint **98 % pour les faux billets** et environ **99,5 % pour
les vrais billets**.

Dans le contexte de l'ONCFM, les **2 faux billets classés comme vrais**
constituent les erreurs les plus sensibles, car ils correspondent à des
billets contrefaits qui ne seraient pas détectés.

Les performances du Random Forest sont, sur ce jeu de test, identiques à
celles obtenues avec la régression logistique et le KNN optimisé en termes
d'accuracy et de matrice de confusion.

## 9.3 Importance des caractéristiques

Le modèle Random Forest permet d'estimer l'importance relative des différentes
caractéristiques géométriques utilisées pour effectuer les prédictions.

Cette analyse permet d'identifier les variables qui contribuent le plus à la
classification des vrais et des faux billets.

In [ ]:
# 9.3.1 Calcul de l'importance des caractéristiques

importance_variables = pd.DataFrame({
    "Variable": X_train.columns,
    "Importance": random_forest_model.feature_importances_
})

importance_variables = importance_variables.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

importance_variables["Importance (%)"] = (
    importance_variables["Importance"] * 100
).round(2)

importance_variables

### Interprétation

L'analyse de l'importance des caractéristiques montre que toutes les variables ne contribuent pas de la même manière aux prédictions du Random Forest.

La variable **`length`** est la plus importante avec **49,57 %** de l'importance totale, suivie de **`margin_low`** avec **31,04 %**.

À elles deux, ces variables représentent environ **80,61 % de l'importance totale du modèle**.

Les autres caractéristiques contribuent dans une moindre mesure :

- **`margin_up` : 10,37 %** ;
- **`height_right` : 4,67 %** ;
- **`height_left` : 3,43 %** ;
- **`diagonal` : 0,92 %**.

Ces résultats sont cohérents avec l'analyse exploratoire réalisée précédemment : les distributions de **`length`** et **`margin_low`** présentaient déjà des différences importantes entre les vrais et les faux billets.

Le Random Forest confirme donc que ces deux caractéristiques jouent un rôle majeur dans la distinction entre les deux catégories de billets.

# 10. Comparaison des modèles et choix du modèle final

Après avoir entraîné et évalué les quatre algorithmes demandés par l'ONCFM,
nous comparons leurs performances afin de sélectionner le modèle le plus
adapté à l'application finale.

Le choix ne repose pas uniquement sur l'accuracy globale. Une attention
particulière est également portée aux erreurs consistant à classer un
**faux billet comme vrai**, compte tenu de l'objectif métier de lutte
contre la contrefaçon.

## 10.1 Tableau comparatif des performances

Nous comparons les performances obtenues sur le jeu de test par les différents
modèles afin d'identifier celui qui offre le meilleur compromis entre performance
globale et détection des faux billets.

Une attention particulière est portée au nombre de **faux billets classés comme vrais**,
car cette erreur représente le risque métier le plus important pour l'ONCFM.

In [ ]:
# 10.1.1 Création du tableau comparatif des modèles

comparaison_modeles = pd.DataFrame({
    "Modèle": [
        "K-means",
        "Régression logistique",
        "KNN optimisé (k=12)",
        "Random Forest"
    ],
    "Accuracy": [
        0.9867,
        0.9900,
        0.9900,
        0.9900
    ],
    "Nombre d'erreurs": [
        4,
        3,
        3,
        3
    ],
    "Faux classés comme vrais": [
        2,
        2,
        2,
        2
    ]
})

comparaison_modeles

## 10.2 Comparaison détaillée des modèles supervisés

Les trois modèles supervisés obtiennent une **accuracy identique de 99 %** sur le jeu de test.

L'accuracy seule ne permet donc pas de sélectionner le modèle final. Nous allons
également prendre en compte :

- la capacité à détecter les faux billets ;
- le nombre de faux billets classés à tort comme vrais ;
- l'interprétabilité du modèle ;
- la possibilité d'obtenir une probabilité de prédiction ;
- la robustesse et la simplicité d'utilisation du modèle.

In [ ]:
# 10.2.1 Tableau d'aide à la décision pour le choix du modèle final

choix_modele = pd.DataFrame({
    "Modèle": [
        "Régression logistique",
        "KNN optimisé (k=12)",
        "Random Forest"
    ],
    "Accuracy": [
        0.9900,
        0.9900,
        0.9900
    ],
    "Erreurs": [
        3,
        3,
        3
    ],
    "Faux classés vrais": [
        2,
        2,
        2
    ],
    "Interprétabilité": [
        "Élevée",
        "Faible",
        "Moyenne"
    ],
    "Probabilités": [
        "Oui",
        "Oui",
        "Oui"
    ]
})

choix_modele

## 10.3 Choix et justification du modèle final

Les trois modèles supervisés présentent d'excellentes performances sur le jeu de test, avec une **accuracy de 99 %** et seulement **3 erreurs sur 300 billets**.

Dans les trois cas, **2 faux billets sont classés à tort comme vrais**. Ce critère ne permet donc pas, à lui seul, de départager les modèles.

Le **KNN optimisé (k = 12)** obtient d'excellentes performances, mais son fonctionnement est moins facilement interprétable et ses prédictions reposent sur la proximité avec les observations du jeu d'entraînement.

Le **Random Forest** obtient également **99 % d'accuracy** et permet d'identifier l'importance des variables. Les résultats montrent notamment que `length` représente environ **49,57 %** de l'importance totale et `margin_low` environ **31,04 %**.

La **régression logistique** obtient elle aussi **99 % d'accuracy**, avec seulement **3 erreurs sur 300 observations**. Elle permet en outre d'obtenir directement une probabilité associée à chaque prédiction, ce qui facilite l'identification des billets pour lesquels le modèle est incertain.

Compte tenu de ses **excellentes performances**, de son **interprétabilité** et de la possibilité d'exploiter les **probabilités de prédiction**, la **régression logistique est retenue comme modèle final**.

## 10.4 Synthèse finale des performances

Le tableau suivant synthétise les résultats obtenus par les quatre approches testées.

Il permet de comparer rapidement leurs performances et de mettre en évidence le
modèle finalement retenu pour la détection des faux billets.

In [ ]:
# 10.4.1 Synthèse finale des performances des modèles

synthese_finale = pd.DataFrame({
    "Modèle": [
        "K-means",
        "Régression logistique",
        "KNN optimisé (k=12)",
        "Random Forest"
    ],
    "Accuracy (%)": [
        98.67,
        99.00,
        99.00,
        99.00
    ],
    "Erreurs sur 300": [
        4,
        3,
        3,
        3
    ],
    "Faux classés vrais": [
        2,
        2,
        2,
        2
    ],
    "Statut": [
        "Comparaison non supervisée",
        "Modèle final retenu",
        "Modèle alternatif",
        "Modèle alternatif"
    ]
})

synthese_finale

# 11. Conclusion générale

L'objectif de cette étude était de développer une méthode permettant de distinguer les **vrais billets des faux billets** à partir de leurs caractéristiques géométriques.

L'analyse exploratoire a montré que le jeu de données contient **1 500 billets et 6 variables explicatives numériques**. La variable `margin_low` présentait **37 valeurs manquantes**, qui ont été traitées par une imputation par la médiane.

L'étude des caractéristiques géométriques a montré que `length` et `margin_low` permettent particulièrement bien de différencier les deux catégories de billets. Cette observation est également confirmée par le Random Forest, qui attribue environ **49,57 % d'importance à `length` et 31,04 % à `margin_low`**.

Quatre approches ont ensuite été étudiées :

- **K-means**, avec une accuracy de **98,67 %** ;
- **Régression logistique**, avec une accuracy de **99 %** ;
- **KNN optimisé (k = 12)**, avec une accuracy de **99 %** ;
- **Random Forest**, avec une accuracy de **99 %**.

Sur ce jeu de test, les trois modèles supervisés obtiennent des **performances identiques en termes d'accuracy et de nombre d'erreurs**. La **régression logistique** est finalement retenue comme modèle final en raison de ses excellentes performances, de son interprétabilité et de sa capacité à fournir **une probabilité associée à chaque prédiction**.

Sur les **300 billets du jeu de test**, elle classe correctement **297 billets** et commet seulement **3 erreurs**. Parmi celles-ci, **2 faux billets sont classés à tort comme vrais**.

Dans une application métier, les probabilités de prédiction pourraient également être utilisées pour identifier les billets pour lesquels le modèle est incertain et les orienter vers un **contrôle complémentaire**.

Le modèle retenu constitue ainsi une solution performante et interprétable pour aider à la détection automatique des billets contrefaits.


# 12. Sauvegarde des éléments du modèle final ONCFM

Cette étape consiste à enregistrer les éléments nécessaires au fonctionnement de l'application de détection des faux billets.

Les éléments sauvegardés sont :

- Le modèle de régression logistique ;
- Le standardiseur des variables ;
- La médiane utilisée pour l'imputation des valeurs manquantes.

Ces fichiers seront réutilisés pour effectuer des prédictions sur de nouveaux billets.


In [ ]:

# 12.1 Sauvegarde du modèle final ONCFM

import joblib

# Enregistrement du modèle de régression logistique
joblib.dump(logistic_model, "modele_ONCFM.pkl")

print("Modèle ONCFM enregistré avec succès !")


In [ ]:

# 12.2 Sauvegarde du standardiseur ONCFM

import joblib

# Enregistrement du standardiseur déjà entraîné
joblib.dump(scaler, "scaler_ONCFM.pkl")

print("Standardiseur ONCFM enregistré avec succès !")


In [ ]:

# 12.3 Sauvegarde de la médiane utilisée pour l'imputation

import joblib

joblib.dump(
    mediane_margin_low_train,
    "mediane_ONCFM.pkl"
)

print("Médiane ONCFM enregistrée avec succès !")
print("Valeur enregistrée :", round(mediane_margin_low_train, 2), "mm")


In [ ]:

# 12.4 Vérification des fichiers nécessaires à l'application ONCFM

import os

fichiers = [
    "modele_ONCFM.pkl",
    "scaler_ONCFM.pkl",
    "mediane_ONCFM.pkl"
]

for fichier in fichiers:
    if os.path.exists(fichier):
        print("OK -", fichier, "est présent")
    else:
        print("ATTENTION -", fichier, "est introuvable")
